# Hierarchischer Ansatz: Vergleich der Klassifikatoren

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Alle hierarchischen Klassifikatoren vergleichen, gemischte Stufen prüfen und die Fehleranalyse aktualisieren (#104).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.base import clone

from awp2.config import CROP_COL, MODEL_DOCS_FIGURES_DIR
from awp2.data import combined_label, load_split
from awp2.evaluation import error_breakdown
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models import HierarchicalClassifier, summarize_hierarchical
from awp2.models.tuned import TUNED_HIERARCHIES, refit_untracked
from awp2.plots import plot_error_breakdown, plot_metric_comparison, save_doc_figure

In [ ]:
split = load_split()
split.X_val.shape

## 1. Alle hierarchischen Klassifikatoren

Die getunten Klassifikatoren der Serie (#87, #98–#103) kommen aus `awp2.models.tuned` und werden
ohne MLflow nachtrainiert (`refit_untracked`); sie reproduzieren die protokollierten
Bewertungsläufe exakt. Nichts wird neu eingestellt. Die Rangfolge gilt nach dem CV-Score der
jeweiligen Suche; die Validierung bestätigt sie nur.

In [ ]:
LABELS = {
    "svm": "RBF-SVM",
    "extra_trees_svm": "Extra Trees → RBF-SVM",
    "extra_trees": "Extra Trees",
    "mlp": "MLP",
    "hist_gradient_boosting": "HistGradientBoosting",
    "logistic_regression": "Logistische Regression",
    "knn": "kNN",
    "plsda": "PLS-DA",
    "lda": "LDA",
}
tuned = {LABELS[name]: factory() for name, factory in TUNED_HIERARCHIES.items()}
results = {LABELS[name]: refit_untracked(tuned[LABELS[name]], name) for name in TUNED_HIERARCHIES}
predictions = {label: result.y_pred for label, result in results.items()}
summaries = {
    label: summarize_hierarchical(result.pipeline, split.X_val, split.y_val)
    for label, result in results.items()
}

In [ ]:
ranking = pd.DataFrame(
    {
        label: {
            "CV": tuned[label].cv_score,
            "BAcc Kultur": summary.metrics.bacc_crop,
            "BAcc Stadium": summary.metrics.bacc_stage,
            "BAcc kombiniert": summary.metrics.bacc_combined,
            **summary.breakdown.model_dump(),
            "Oracle-Kultur": summary.oracle.bacc_combined,
            "Tuning-Lauf": tuned[label].tuning_run,
        }
        for label, summary in summaries.items()
    }
).T.sort_values("CV", ascending=False)
ranking

In [ ]:
ranking_ax = plot_metric_comparison(
    {label: summary.metrics for label, summary in summaries.items()}, metrics=("bacc_combined",)
)
ranking_ax.set_title("Hierarchischer Ansatz: alle Klassifikatoren (Validierung)")
ranking_ax.legend(fontsize="x-small", ncol=3, loc="lower center")
save_doc_figure(ranking_ax.figure, "hierarchical_vergleich", MODEL_DOCS_FIGURES_DIR)

In [ ]:
breakdown_ax = plot_error_breakdown(
    {label: error_breakdown(split.y_val, predictions[label]) for label in ranking.index}
)
breakdown_ax.set_title("Fehlerzerlegung aller hierarchischen Klassifikatoren")
save_doc_figure(breakdown_ax.figure, "hierarchical_vergleich_zerlegung", MODEL_DOCS_FIGURES_DIR)

## 2. Hilft eine Mischung der Stufen?

Jede Stufe kann einen anderen Klassifikator nutzen. Eine CV-Suche kombiniert die besten
Einstellungen der schnellen Klassifikatoren mit Klassengewichten bzw. gleichen Priors auf beiden
Stufen. Nur wenn eine Mischung die reine SVM in der CV schlägt, wird sie einmal bewertet.

In [ ]:
MIXED_LEVELS = ["RBF-SVM", "Logistische Regression", "kNN"]
STAGE_ONLY = ["LDA"]
crop_candidates = [tuned[label].model.crop_estimator for label in MIXED_LEVELS]
stage_candidates = [tuned[label].model.stage_estimator for label in MIXED_LEVELS + STAGE_ONLY]
preprocessing = tuned["RBF-SVM"].preprocessing
mixed = tune(
    HierarchicalClassifier(crop_candidates[0], stage_candidates[0]),
    TuneConfig(
        name="hierarchical_mixed_search",
        approach="hierarchical",
        description="Mischung der Stufen: getunte Kultur- und Stadium-Modelle aus #87, #98, "
        "#99, #101 per CV kombiniert.",
        preprocessing=preprocessing,
        param_grid={"crop_estimator": crop_candidates, "stage_estimator": stage_candidates},
    ),
)
mixed_table = mixed.candidates.assign(
    Kultur=[type(p["crop_estimator"]).__name__ for p in mixed.candidates["params"]],
    Stadium=[type(p["stage_estimator"]).__name__ for p in mixed.candidates["params"]],
)[["Kultur", "Stadium", "mean", "std", "rank"]].sort_values("rank")
mixed_table.round(4)

In [ ]:
# the SVM is the first candidate on both levels; compare positions, not object identity
best_is_pure_svm = (
    crop_candidates.index(mixed.best_params["crop_estimator"]) == 0
    and stage_candidates.index(mixed.best_params["stage_estimator"]) == 0
)
if best_is_pure_svm:
    mixed_result = {"Entscheidung": "RBF-SVM auf beiden Stufen bleibt; kein neuer Lauf"}
else:
    mixed_result = run(
        mixed.best_model,
        RunConfig(
            name="hierarchical_mixed",
            approach="hierarchical",
            description="Beste Mischung der Stufen aus hierarchical_mixed_search, einmal bewertet.",
            preprocessing=preprocessing,
            tuning_run=mixed.run_id,
        ),
    ).metrics.model_dump()
mixed_result, mixed.run_id

## 3. Ergänzen sich die Klassifikatoren?

Wie viele Zeilen trifft *mindestens ein* Klassifikator? Der Abstand zur SVM ist eine Obergrenze
dessen, was ein Ensemble gewinnen könnte – ein Hinweis, keine erreichbare Zahl. Gerechnet wird
zuerst auf Out-of-fold-Vorhersagen des Trainingsteils (Grundlage für die Wahl von
Ensemble-Kandidaten), die Validierung dient nur als Gegenprobe. Achtung: Extra Trees → RBF-SVM und
RBF-SVM teilen dieselbe Stadium-SVM, die Modelle sind also nicht unabhängig.

In [ ]:
oof_predictions = {
    label: out_of_fold_predictions(
        clone(t.model),
        RunConfig(
            name="hierarchical_oof",
            approach="hierarchical",
            preprocessing=t.preprocessing,
            balance_samples=t.balance_samples,
            track=False,
        ),
    )
    for label, t in tuned.items()
}

In [ ]:
def complementarity(predicted: dict[str, pd.DataFrame], truth: pd.DataFrame) -> pd.Series:
    correct = pd.DataFrame(
        {label: combined_label(pred) == combined_label(truth) for label, pred in predicted.items()}
    )
    crop_correct = pd.DataFrame(
        {label: pred[CROP_COL] == truth[CROP_COL] for label, pred in predicted.items()}
    )
    svm_crop_wrong = ~crop_correct["RBF-SVM"]
    return pd.Series(
        {
            "RBF-SVM richtig": correct["RBF-SVM"].mean(),
            "mindestens ein Klassifikator richtig": correct.any(axis=1).mean(),
            "SVM falsch, mindestens ein anderer richtig": (
                ~correct["RBF-SVM"] & correct.any(axis=1)
            ).mean(),
            "Kultur: SVM falsch (Zeilen)": svm_crop_wrong.sum(),
            "Kultur: SVM falsch, ein anderer richtig (Zeilen)": (
                svm_crop_wrong & crop_correct.drop(columns="RBF-SVM").any(axis=1)
            ).sum(),
        }
    )


pd.DataFrame(
    {
        "Training (out-of-fold)": complementarity(oof_predictions, split.y_train),
        "Validierung": complementarity(predictions, split.y_val),
    }
).round(3)

In [ ]:
oof_crop_correct = pd.DataFrame(
    {label: pred[CROP_COL] == split.y_train[CROP_COL] for label, pred in oof_predictions.items()}
)
svm_misses = ~oof_crop_correct["RBF-SVM"]
pd.Series(
    {
        label: (svm_misses & oof_crop_correct[label]).sum()
        for label in oof_crop_correct.columns.drop("RBF-SVM")
    },
    name="Kulturfehler der SVM, die dieses Modell trifft (Training, out-of-fold)",
).sort_values(ascending=False)

## Befund

- Nach CV ist die RBF-SVM auf beiden Stufen der beste hierarchische Klassifikator (CV 0.839,
  Validierung 0.844); jede Mischung der Stufen ist in der CV schlechter.
- Lineare Modelle verlieren vor allem auf der Kulturstufe, MLP und kNN auch beim Stadium.
- Die Klassifikatoren ergänzen sich – auf dem Trainingsteil (out-of-fold) trifft mindestens einer
  96,2 % der Zeilen (SVM 87,7 %). Die komplementärsten Modelle sind dort kNN, logistische
  Regression und Extra Trees: Kandidaten für einen Ensemble-Versuch per CV (#113).